# Experimento 3 - Drop Insulin, imputação por mediana e RobustScaler

## Dataset de treino

In [60]:
import pandas as pd
from sklearn.preprocessing import RobustScaler

NORMALIZE_COLS = ['Glucose', 'BloodPressure', 'SkinThickness', 'BMI', 'DiabetesPedigreeFunction']

df = pd.read_csv("../../diabetes_dataset.csv")
df.shape


(572, 9)

### 1. Dropando `Insulin`

65% dos registros não tinham esse valor e, ao tentar imputar por mediana, criava um pico artificial enorme na coluna, então ela sai do dataset.

In [61]:
df = df.drop(columns=['Insulin'])
df.columns.tolist()


['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age',
 'Outcome']

### 2. Imputando NaN com a mediana

Guardamos as medianas calculadas aqui — elas também vão ser usadas no
dataset de previsão, pra não usar nenhuma estatística fora do treino.

In [62]:
medianas = df.median(numeric_only=True)
df = df.fillna(medianas)
df.isna().sum()

# medianas

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

### 3. Normalizando com RobustScaler

Só em `Glucose`, `BloodPressure`, `SkinThickness`, `BMI`,
`DiabetesPedigreeFunction`. `Pregnancies` e `Age` ficam brutas (são
contagens/anos, não medidas contínuas como as outras), e `Outcome` não é
tocado. O `scaler` é ajustado aqui e reaproveitado no dataset de previsão.

In [63]:
scaler = RobustScaler()
df[NORMALIZE_COLS] = scaler.fit_transform(df[NORMALIZE_COLS])
df.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,0.789809,0.000000,1.25,0.181818,0.764547,50,1
1,1,-0.815287,-0.428571,-0.25,-0.613636,0.017591,31,0
2,8,1.681529,-0.571429,0.00,-0.988636,0.886333,32,1
3,0,0.509554,-2.285714,1.25,1.261364,5.259811,33,1
4,5,-0.025478,0.142857,0.00,-0.727273,-0.388363,30,0


### 4. Exportando

In [64]:
df.to_csv("diabetes-tratado-ex3.csv", index=False)
print("treino exportado com", len(df), "registros e colunas:", list(df.columns))


treino exportado com 572 registros e colunas: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']


## Dataset de previsão (`diabetes_app.csv`)

Mesmas transformações do treino, mas reaproveitando as medianas e o
`scaler` já ajustados acima (sem `fit` de novo) — treino e previsão
precisam ficar na mesma escala pro KNN funcionar.

In [65]:
df_app = pd.read_csv("../../diabetes_app.csv")
df_app = df_app.drop(columns=['Insulin'])
df_app = df_app.fillna(medianas)
df_app[NORMALIZE_COLS] = scaler.transform(df_app[NORMALIZE_COLS])
df_app.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,BMI,DiabetesPedigreeFunction,Age
0,3,0.305732,-0.571429,-0.25,-0.636364,-0.339648,28
1,3,1.885350,-0.285714,-3.75,-0.125000,-0.123139,34
2,15,0.484076,-0.142857,0.50,0.579545,-0.518268,43
3,1,-0.203822,-0.857143,-5.50,-0.750000,1.630582,21
4,8,0.866242,0.428571,0.50,1.238636,0.464141,36


In [66]:
df_app.to_csv("diabetes_app-tratado-ex3.csv", index=False)
print("app exportado com", len(df_app), "registros e colunas:", list(df_app.columns))


app exportado com 196 registros e colunas: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'BMI', 'DiabetesPedigreeFunction', 'Age']


# Resultado

O servidor de previsão retornou:
```
 - Resposta do servidor:
 {"status":"success","dev_key":"Omicron","accuracy":0.6428571428571429,"old_accuracy":0.60204081632653}
 ```